In [1]:
# Vectorizer --> Bag of Words --> CountVectorizer
# Neural Network --> nn.Linear
# Activation Function --> ReLu
# Regularization --> Dropout
# Optimizer --> Adam
# Loss Function --> BCEWithLogitsLoss

In [3]:
# =========================
# IMPORTS
# =========================
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import joblib as jb


In [4]:
# =========================
# CONFIG (EASY TO CHANGE)
# =========================
EPOCHS = 10
BATCH_SIZE = 64

# Try multiple configs automatically
WIDTHS = [16,32,64,128]
LAYERS = [1,2,3,4,5]

DROPOUT = 0.1
LEARNING_RATE = 1e-3
L2 = 1e-4  # weight decay

In [5]:
!nvidia-smi
%ls
print(torch.cuda.is_available())

Thu May 14 17:33:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [18]:
# =========================
# LOAD DATA
# =========================
if torch.cuda.is_available():
    df = pd.read_csv("../../assets/cleaned_data/scikit_cleaned.csv")
else:
    df = pd.read_csv("/content/drive/MyDrive/scikit_cleaned.csv")

texts = (df['sender'].fillna('') + ' ' + df['receiver'].fillna('') + ' ' + df['date'].fillna('') + ' ' + df['subject'].fillna('') + ' ' + df['body'].fillna(''))
labels = df['label']

In [19]:
# =========================
# SPLIT: 60 / 20 / 20
# =========================
X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42  # 0.25 of 80% = 20%
)

In [20]:
# =========================
# VECTORIZER (BAG OF WORDS)
# =========================
vectorizer = CountVectorizer(max_features=10000)

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_val_vec = vectorizer.transform(X_val).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

# Convert to tensors
X_train_tensor = torch.tensor(X_train_vec, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_vec, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_vec, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

input_size = X_train_tensor.shape[1]

In [21]:
# =========================
# MODEL DEFINITION
# =========================
class PhishingNN(nn.Module):
    def __init__(self, input_size, width, depth):
        super().__init__()

        layers = []

        # input layer
        layers.append(nn.Linear(input_size, width))
        layers.append(nn.GELU())
        layers.append(nn.Dropout(DROPOUT))

        # hidden layers
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(DROPOUT))

        # output layer
        layers.append(nn.Linear(width, 1))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [22]:
# =========================
# TRAIN FUNCTION
# =========================
def train_model(model, X_train, y_train, X_val, y_val):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=L2)

    for epoch in range(EPOCHS):
        model.train()

        outputs = model(X_train).squeeze()
        loss = criterion(outputs, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val).squeeze()
            val_preds = (torch.sigmoid(val_outputs) > 0.5).int()
            val_acc = accuracy_score(y_val, val_preds)

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")

    return val_acc


In [23]:
# =========================
# EXPERIMENT LOOP
# =========================
results = []

for width in WIDTHS:
    for depth in LAYERS:
        print(f"\nTraining model: width={width}, depth={depth}")

        model = PhishingNN(input_size, width, depth)

        val_acc = train_model(
            model,
            X_train_tensor,
            y_train_tensor,
            X_val_tensor,
            y_val_tensor
        )

        results.append({
            "width": width,
            "depth": depth,
            "val_acc": val_acc,
            "model": model
        })


Training model: width=64, depth=1
Epoch 1/10 | Loss: 0.6982 | Val Acc: 0.7618
Epoch 2/10 | Loss: 0.6237 | Val Acc: 0.7747
Epoch 3/10 | Loss: 0.5846 | Val Acc: 0.8123
Epoch 4/10 | Loss: 0.5516 | Val Acc: 0.8416
Epoch 5/10 | Loss: 0.5204 | Val Acc: 0.8745
Epoch 6/10 | Loss: 0.4907 | Val Acc: 0.9052
Epoch 7/10 | Loss: 0.4633 | Val Acc: 0.9309
Epoch 8/10 | Loss: 0.4371 | Val Acc: 0.9475
Epoch 9/10 | Loss: 0.4117 | Val Acc: 0.9573
Epoch 10/10 | Loss: 0.3877 | Val Acc: 0.9633

Training model: width=64, depth=2
Epoch 1/10 | Loss: 0.6971 | Val Acc: 0.6282
Epoch 2/10 | Loss: 0.6694 | Val Acc: 0.7764
Epoch 3/10 | Loss: 0.6432 | Val Acc: 0.8250
Epoch 4/10 | Loss: 0.6180 | Val Acc: 0.8480
Epoch 5/10 | Loss: 0.5941 | Val Acc: 0.8580
Epoch 6/10 | Loss: 0.5705 | Val Acc: 0.8669
Epoch 7/10 | Loss: 0.5465 | Val Acc: 0.8801
Epoch 8/10 | Loss: 0.5219 | Val Acc: 0.9008
Epoch 9/10 | Loss: 0.4954 | Val Acc: 0.9248
Epoch 10/10 | Loss: 0.4672 | Val Acc: 0.9446

Training model: width=64, depth=4
Epoch 1/10 | 

In [24]:
# =========================
# BEST MODEL SELECTION
# =========================
best = max(results, key=lambda x: x["val_acc"])

print("\nBest Model:")
print(best["width"], best["depth"], best["val_acc"])

best_model = best["model"]


Best Model:
1024 4 0.9807964943936074


In [ ]:
def evaluate_on_test():
    best_model.eval()
    with torch.no_grad():
        outputs = best_model(X_test_tensor).squeeze()
        preds = (torch.sigmoid(outputs) > 0.5).int()
        acc = accuracy_score(y_test, preds)

    

    saved_metadata = {
        "width": best["width"],
        "depth": best["depth"],
        "dropout": DROPOUT,
        "input_size": input_size,
        "val_acc": best["val_acc"]
    }

    if torch.cuda.is_available():
        torch.save(best_model.state_dict(), "../../assets/models/deeplearning_model.pth")
        jb.dump(vectorizer, "../../assets/vectorizers/deeplearning_vectorizer.pkl")
        torch.save(saved_metadata, "../../assets/metadata/deeplearning_metadata.pth")
    else:
        torch.save(best_model.state_dict(), "/content/drive/MyDrive/PhishingModels/deeplearning_model.pth")
        jb.dump(vectorizer, "/content/drive/MyDrive/PhishingModels/deeplearning_vectorizer.pkl")
        torch.save(saved_metadata, "/content/drive/MyDrive/PhishingModels/deeplearning_metadata.pth")

    print(f"TEST ACCURACY: {acc:.4f}")

In [ ]:
#commented so I don't accidentally run

#evaluate_on_test()

TEST ACCURACY: 0.9805
